# Colab 2 · Orchestration Patterns in Depth
### Day 19 — Agent Orchestration with AutoGen Studio & Semantic Kernel

Colab 1 used the simplest pattern (RoundRobin). Now you'll wire the **same research team four different ways** and watch how the *control flow* changes — then peek at the **same idea in Semantic Kernel**.

**You will build:**
1. **SelectorGroupChat** — an LLM decides who speaks next.
2. **Swarm** — agents hand off to each other directly.
3. **GraphFlow** — a deterministic researcher → writer → reviewer graph.
4. A **function tool** the researcher calls to delegate real work.
5. A **Semantic Kernel** sequential-orchestration mini-example.

⏱️ ~60 min including the extension tasks at the end.

> The patterns are the lesson. AutoGen, Semantic Kernel and the Microsoft Agent Framework all expose this same family — RoundRobin/Sequential, Selector/GroupChat, Swarm/Handoff, Graph, Magentic.

## 0 · Setup

In [1]:
%pip install -q -U "autogen-agentchat" "autogen-ext[openai]"
print("AutoGen installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.3/119.3 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.9/101.9 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.4/331.4 kB 7.3 MB/s eta 0:00:00
AutoGen installed.


In [2]:
import os
from getpass import getpass
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Paste your OpenAI API key: ")

from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.conditions import TextMentionTermination, MaxMessageTermination
from autogen_agentchat.ui import Console

model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
print("Ready.")

Paste your OpenAI API key: ··········
Ready.


### Three specialists we'll reuse

Notice the **descriptions** — Selector and Swarm route work based on them, so they have to be specific and non-overlapping.

In [3]:
def make_specialists():
    planner = AssistantAgent(
        name="planner",
        model_client=model_client,
        description="Breaks a topic into 2-3 concrete sub-questions to research.",
        system_message="You plan research. Given a topic, list 2-3 specific sub-questions. Keep it short.",
    )
    researcher = AssistantAgent(
        name="researcher",
        model_client=model_client,
        description="Answers factual sub-questions with concise bullet points.",
        system_message="You answer the planner's sub-questions with short factual bullets.",
    )
    writer = AssistantAgent(
        name="writer",
        model_client=model_client,
        description="Turns research bullets into a tight 4-sentence summary, ending with APPROVE.",
        system_message="Write a tight 4-sentence summary from the research. End your message with APPROVE.",
    )
    return planner, researcher, writer

print("Specialist factory ready.")

Specialist factory ready.


## 1 · SelectorGroupChat — let an LLM route

Instead of a fixed order, a **SelectorGroupChat** uses a model to pick *who should act next* based on the conversation and each agent's `description`. Good when the next best speaker depends on what just happened.

Key knobs: it needs its own `model_client` to do the routing, and `allow_repeated_speaker=False` stops one agent from monopolising the floor.

In [4]:
from autogen_agentchat.teams import SelectorGroupChat

planner, researcher, writer = make_specialists()
termination = TextMentionTermination("APPROVE") | MaxMessageTermination(8)

selector_team = SelectorGroupChat(
    participants=[planner, researcher, writer],
    model_client=model_client,        # the "router" brain
    termination_condition=termination,
    allow_repeated_speaker=False,
)

await Console(selector_team.run_stream(
    task="Topic: why are reusable cups better than disposable ones?"
))

---------- TextMessage (user) ----------
Topic: why are reusable cups better than disposable ones?
---------- TextMessage (planner) ----------
1. What environmental impacts do reusable cups have compared to disposable cups in terms of waste reduction and resource consumption?  
2. How does the life cycle cost of reusable cups compare to that of disposable cups over time?  
3. What are the health and safety considerations associated with using reusable cups versus disposable ones?
---------- TextMessage (researcher) ----------
1. **Environmental Impacts**  
   - **Waste Reduction**: Reusable cups significantly reduce landfill waste by being used multiple times, unlike disposable cups that contribute to single-use waste.  
   - **Resource Consumption**: The production of disposable cups requires more raw materials (wood pulp, plastics) which leads to deforestation and higher energy consumption compared to a single reusable cup.  
   - **Carbon Footprint**: Over time, reusable cups result

TaskResult(messages=[TextMessage(id='d88542f5-1d0c-43d0-b6bc-a8193083e359', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 6, 22, 11, 41, 6, 163206, tzinfo=datetime.timezone.utc), content='Topic: why are reusable cups better than disposable ones?', type='TextMessage'), TextMessage(id='123040df-4f44-436c-9a67-b94796c05a0c', source='planner', models_usage=RequestUsage(prompt_tokens=45, completion_tokens=62), metadata={}, created_at=datetime.datetime(2026, 6, 22, 11, 41, 10, 200065, tzinfo=datetime.timezone.utc), content='1. What environmental impacts do reusable cups have compared to disposable cups in terms of waste reduction and resource consumption?  \n2. How does the life cycle cost of reusable cups compare to that of disposable cups over time?  \n3. What are the health and safety considerations associated with using reusable cups versus disposable ones?', type='TextMessage'), TextMessage(id='637fd870-0329-46fb-acf7-e3677b61ee5f', source='researcher

Look at the speaker order in the transcript — it was **chosen at runtime**, not fixed. That's the difference from RoundRobin.

## 2 · Swarm — agents hand off to each other

In a **Swarm**, control is decentralised: each agent declares who it can **hand off** to via `handoffs=[...]`, and passes control with a handoff message. There's no central router — the agents themselves decide.

We'll build a tiny triage flow: a `triage` agent routes to either `billing` or `tech`, and those can hand back to the user when done.

In [5]:
from autogen_agentchat.teams import Swarm
from autogen_agentchat.conditions import HandoffTermination

triage = AssistantAgent(
    name="triage",
    model_client=model_client,
    handoffs=["billing", "tech"],
    description="Front desk: routes the user to the right specialist.",
    system_message="Decide if the request is about billing or tech, then hand off to that agent.",
)
billing = AssistantAgent(
    name="billing",
    model_client=model_client,
    handoffs=["triage"],
    description="Handles billing and refund questions.",
    system_message="Answer the billing question. If it's not billing, hand back to triage.",
)
tech = AssistantAgent(
    name="tech",
    model_client=model_client,
    handoffs=["triage"],
    description="Handles technical troubleshooting.",
    system_message="Answer the tech question concisely, then say DONE.",
)

swarm = Swarm(
    participants=[triage, billing, tech],          # Swarm starts with the first agent
    termination_condition=TextMentionTermination("DONE") | MaxMessageTermination(8),
)

await Console(swarm.run_stream(task="My app keeps crashing when I open the camera."))

---------- TextMessage (user) ----------
My app keeps crashing when I open the camera.
---------- ToolCallRequestEvent (triage) ----------
[FunctionCall(id='call_Og39ycsGHV2P4rGkVpDwRG1e', arguments='{}', name='transfer_to_tech')]
---------- ToolCallExecutionEvent (triage) ----------
[FunctionExecutionResult(content='Transferred to tech, adopting the role of tech immediately.', name='transfer_to_tech', call_id='call_Og39ycsGHV2P4rGkVpDwRG1e', is_error=False)]
---------- HandoffMessage (triage) ----------
Transferred to tech, adopting the role of tech immediately.
---------- ToolCallRequestEvent (tech) ----------
[FunctionCall(id='call_bohy599ovrXZu0ucpj9f4Ojk', arguments='{}', name='transfer_to_triage')]
---------- ToolCallExecutionEvent (tech) ----------
[FunctionExecutionResult(content='Transferred to triage, adopting the role of triage immediately.', name='transfer_to_triage', call_id='call_bohy599ovrXZu0ucpj9f4Ojk', is_error=False)]
---------- HandoffMessage (tech) ----------
Trans

TaskResult(messages=[TextMessage(id='0a3cc216-2905-4390-bea4-20e37aaf0cc0', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 6, 22, 11, 41, 17, 355185, tzinfo=datetime.timezone.utc), content='My app keeps crashing when I open the camera.', type='TextMessage'), ToolCallRequestEvent(id='db7f440d-72d7-4019-b4db-5ae84f12072e', source='triage', models_usage=RequestUsage(prompt_tokens=82, completion_tokens=12), metadata={}, created_at=datetime.datetime(2026, 6, 22, 11, 41, 17, 981528, tzinfo=datetime.timezone.utc), content=[FunctionCall(id='call_Og39ycsGHV2P4rGkVpDwRG1e', arguments='{}', name='transfer_to_tech')], type='ToolCallRequestEvent'), ToolCallExecutionEvent(id='0480b681-c110-4f9e-983e-fef5cbc37693', source='triage', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 6, 22, 11, 41, 17, 983655, tzinfo=datetime.timezone.utc), content=[FunctionExecutionResult(content='Transferred to tech, adopting the role of tech immediately.', name='tra

Watch for the **HandoffMessage** in the transcript — that's one agent explicitly delegating to another. `HandoffTermination(target="user")` is another common stop condition when an agent hands control back to a human.

## 3 · GraphFlow — a deterministic workflow

When you need the **same path every time** (auditable, reproducible), use **GraphFlow**. You declare nodes and directed edges with `DiGraphBuilder`; execution follows the graph exactly.

Here: `planner → researcher → writer`, a fixed pipeline with no LLM routing brain.

In [6]:
from autogen_agentchat.teams import DiGraphBuilder, GraphFlow

planner, researcher, writer = make_specialists()

builder = DiGraphBuilder()
builder.add_node(planner).add_node(researcher).add_node(writer)
builder.add_edge(planner, researcher).add_edge(researcher, writer)
graph = builder.build()

flow = GraphFlow(
    participants=builder.get_participants(),
    graph=graph,
)

await Console(flow.run_stream(task="Topic: the benefits of cycling to work."))

---------- TextMessage (user) ----------
Topic: the benefits of cycling to work.
---------- TextMessage (planner) ----------
1. How does cycling to work impact employee productivity and job satisfaction?
2. What are the environmental benefits of increased cycling as a mode of transportation to workplaces?
3. How does commuting by bike affect physical health and wellness among employees?
---------- TextMessage (researcher) ----------
1. **Employee Productivity and Job Satisfaction:**
   - Increases energy levels and reduces fatigue.
   - Improves focus and cognitive function.
   - Enhances mood, reducing stress and anxiety.
   - Creates a sense of accomplishment and boosts morale.

2. **Environmental Benefits:**
   - Reduces greenhouse gas emissions compared to motor vehicle commuting.
   - Decreases air pollution and improves air quality.
   - Lowers traffic congestion, leading to less urban sprawl.
   - Conserves energy by utilizing human power instead of fossil fuels.

3. **Physical 

TaskResult(messages=[TextMessage(id='7057e963-8ec2-44e2-a775-27ea9b6b1dc3', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 6, 22, 11, 41, 27, 771386, tzinfo=datetime.timezone.utc), content='Topic: the benefits of cycling to work.', type='TextMessage'), TextMessage(id='9d6d0496-6ead-4aed-9a19-5f21df2bc22f', source='planner', models_usage=RequestUsage(prompt_tokens=43, completion_tokens=47), metadata={}, created_at=datetime.datetime(2026, 6, 22, 11, 41, 29, 35630, tzinfo=datetime.timezone.utc), content='1. How does cycling to work impact employee productivity and job satisfaction?\n2. What are the environmental benefits of increased cycling as a mode of transportation to workplaces?\n3. How does commuting by bike affect physical health and wellness among employees?', type='TextMessage'), TextMessage(id='3f692c60-b72e-4012-a173-a19bbb97c3b2', source='researcher', models_usage=RequestUsage(prompt_tokens=87, completion_tokens=171), metadata={}, created_at=

GraphFlow gives you **determinism**: the order is guaranteed by the graph, not decided by a model. That's exactly what you want for a compliance-sensitive or repeatable pipeline.

## 4 · A function tool the researcher can call

Delegation isn't only agent-to-agent — an agent can delegate to **code** via a tool. Define a plain Python function, pass it in `tools=[...]`, and the agent will call it when useful. (Here it's a stub; swap in a real search API at home.)

In [7]:
def web_search(query: str) -> str:
    """Look up a query and return a short text snippet. (Stub for the workshop.)"""
    canned = {
        "reusable cup co2": "A reusable cup typically breaks even vs. disposables after ~20-100 uses.",
        "default": "No exact match; returning a generic note that reusable goods amortise their footprint with use.",
    }
    return canned.get(query.lower().strip(), canned["default"])

researcher_with_tool = AssistantAgent(
    name="researcher",
    model_client=model_client,
    tools=[web_search],
    description="Researches facts, calling web_search when it needs evidence.",
    system_message="Use the web_search tool to find a figure, then report it in one bullet. End with APPROVE.",
)

from autogen_agentchat.teams import RoundRobinGroupChat
tool_team = RoundRobinGroupChat(
    [researcher_with_tool],
    termination_condition=TextMentionTermination("APPROVE") | MaxMessageTermination(4),
)
await Console(tool_team.run_stream(task="Find a figure on reusable cup CO2 break-even and report it."))

---------- TextMessage (user) ----------
Find a figure on reusable cup CO2 break-even and report it.
---------- ToolCallRequestEvent (researcher) ----------
[FunctionCall(id='call_502qZsCIlpB6wL3wl0i0z4GF', arguments='{"query":"reusable cup CO2 break-even figure"}', name='web_search')]
---------- ToolCallExecutionEvent (researcher) ----------
[FunctionExecutionResult(content='No exact match; returning a generic note that reusable goods amortise their footprint with use.', name='web_search', call_id='call_502qZsCIlpB6wL3wl0i0z4GF', is_error=False)]
---------- ToolCallSummaryMessage (researcher) ----------
No exact match; returning a generic note that reusable goods amortise their footprint with use.
---------- TextMessage (researcher) ----------
- Reusable cups can amortize their carbon footprint with repeated use, though exact figures can vary depending on materials and usage patterns. APPROVE.


TaskResult(messages=[TextMessage(id='53c2e184-06f9-49a7-a41f-b344f6d8b994', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 6, 22, 11, 41, 33, 522919, tzinfo=datetime.timezone.utc), content='Find a figure on reusable cup CO2 break-even and report it.', type='TextMessage'), ToolCallRequestEvent(id='0fe2a56e-4f4a-4807-97e9-812cbd6fd73c', source='researcher', models_usage=RequestUsage(prompt_tokens=97, completion_tokens=21), metadata={}, created_at=datetime.datetime(2026, 6, 22, 11, 41, 36, 16669, tzinfo=datetime.timezone.utc), content=[FunctionCall(id='call_502qZsCIlpB6wL3wl0i0z4GF', arguments='{"query":"reusable cup CO2 break-even figure"}', name='web_search')], type='ToolCallRequestEvent'), ToolCallExecutionEvent(id='927405d1-de73-4a0b-bfd3-1e17a2f075b8', source='researcher', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 6, 22, 11, 41, 36, 34346, tzinfo=datetime.timezone.utc), content=[FunctionExecutionResult(content='No exact matc

The transcript shows a **ToolCall** and its result — the agent delegated part of its job to your function. In production you'd scope tools to least privilege and validate their arguments.

## 5 · The same idea in Semantic Kernel

Semantic Kernel expresses these patterns too — its **Sequential** orchestration is the SK analogue of RoundRobin/GraphFlow-in-a-line. The code below shows the *shape* of SK agent orchestration.

> ⚠️ SK's agent-orchestration API is newer and evolving (and SK is in maintenance mode heading into the Microsoft Agent Framework). If an import path has moved, check the official Semantic Kernel docs — the **concept** is what transfers, not the exact symbol names.

In [8]:
%pip install -q -U semantic-kernel
print("Semantic Kernel installed.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.6/74.6 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.7/91.7 kB 5.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 3.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 929.3/929.3 kB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.2/93.2 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 217.9/217.9 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.2/115.2 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.1/192.1 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.6/106.6 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 MB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [9]:
# The shape of an SK sequential orchestration: two agents, output of one feeds the next.
# Wrapped in try/except because SK's orchestration symbols move between versions.
import asyncio

try:
    from semantic_kernel.agents import ChatCompletionAgent
    from semantic_kernel.agents.orchestration.sequential import SequentialOrchestration
    from semantic_kernel.agents.runtime import InProcessRuntime
    from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion

    service = OpenAIChatCompletion(ai_model_id="gpt-4o-mini")

    sk_writer = ChatCompletionAgent(
        name="writer", service=service,
        instructions="Write one short paragraph on the given topic.",
    )
    sk_editor = ChatCompletionAgent(
        name="editor", service=service,
        instructions="Tighten the paragraph you receive into two crisp sentences.",
    )

    orchestration = SequentialOrchestration(members=[sk_writer, sk_editor])
    runtime = InProcessRuntime()
    runtime.start()

    result = await orchestration.invoke(
        task="The benefits of walking meetings.", runtime=runtime
    )
    print(await result.get())
    await runtime.stop_when_idle()

except Exception as e:
    print("SK orchestration symbols may have moved in your installed version.")
    print("Concept: members=[writer, editor] run in sequence, output -> input.")
    print("Check https://learn.microsoft.com/semantic-kernel for the current API.")
    print("Error was:", type(e).__name__, e)

Walking meetings combine physical activity with productive collaboration, boosting creativity and problem-solving while reducing stress. They foster open communication and camaraderie among team members, enhancing well-being and team productivity.


Notice the **identical mental model**: a list of agents, run in order, each one's output feeding the next. RoundRobin (AutoGen) ≈ Sequential (SK) ≈ a linear GraphFlow. Learn it once.

---
## 🚀 Extension tasks

### Extension 1 — Write a custom selector function
`SelectorGroupChat` accepts a `selector_func` that overrides the LLM router with your own logic. Write a function that **forces** `planner` to go first, then lets the model choose. (Signature: it receives the message history and returns the next speaker's name, or `None` to defer to the model.)

### Extension 2 — Add a conditional GraphFlow edge
Extend the graph from §3 with a **reviewer** node and a **conditional edge**: if the writer's output contains the word `REVISE`, loop back to the writer; otherwise finish. Use `DiGraphBuilder`'s conditional-edge support and a stop condition so it can't loop forever.

### Extension 3 — Nest a team inside a graph node
A node in a GraphFlow can itself be a **team**. Replace the single `researcher` node with a 2-agent `RoundRobinGroupChat` (researcher + fact-checker) and wire that team in as one node. This is *composition*: patterns nest inside patterns.

Scaffolds below.

In [15]:
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.conditions import TextMentionTermination, MaxMessageTermination
from autogen_agentchat.ui import Console

model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
print("Model client re-initialized.")

Model client re-initialized.


In [16]:
def make_specialists():
    planner = AssistantAgent(
        name="planner",
        model_client=model_client,
        description="Breaks a topic into 2-3 concrete sub-questions to research.",
        system_message="You plan research. Given a topic, list 2-3 specific sub-questions. Keep it short.",
    )
    researcher = AssistantAgent(
        name="researcher",
        model_client=model_client,
        description="Answers factual sub-questions with concise bullet points.",
        system_message="You answer the planner's sub-questions with short factual bullets.",
    )
    writer = AssistantAgent(
        name="writer",
        model_client=model_client,
        description="Turns research bullets into a tight 4-sentence summary, ending with APPROVE.",
        system_message="Write a tight 4-sentence summary from the research. End your message with APPROVE.",
    )
    return planner, researcher, writer

print("Specialist factory ready.")

Specialist factory ready.


In [17]:
# === Extension 1 scaffold: custom selector_func ===
from autogen_agentchat.teams import SelectorGroupChat

def force_planner_first(messages):
    # Return an agent name (str) to force a speaker, or None to let the model decide.
    if len(messages) <= 1:
        return "planner"
    return None

planner, researcher, writer = make_specialists()
team = SelectorGroupChat(
    participants=[planner, researcher, writer],
    model_client=model_client,
    selector_func=force_planner_first,
    termination_condition=TextMentionTermination("APPROVE") | MaxMessageTermination(8),
)
await Console(team.run_stream(task="Topic: benefits of a standing desk."))

---------- TextMessage (user) ----------
Topic: benefits of a standing desk.
---------- TextMessage (planner) ----------
1. How does using a standing desk impact overall productivity and focus in the workplace?  
2. What are the physical health benefits associated with prolonged standing versus sitting?  
3. How do standing desks affect energy levels and fatigue throughout the workday?
---------- TextMessage (researcher) ----------
1. **Impact on Productivity and Focus:**
   - Increased energy and alertness.
   - Potential for improved concentration levels.
   - May reduce distractions caused by discomfort from prolonged sitting.
   - Some studies suggest enhanced task performance.

2. **Physical Health Benefits:**
   - Reduces risk of obesity and weight gain.
   - Lowers chances of developing chronic conditions (e.g., heart disease, diabetes).
   - Alleviates back and neck pain.
   - Encourages better posture and core strength.

3. **Effects on Energy Levels and Fatigue:**
   - Standi

TaskResult(messages=[TextMessage(id='6e186bc4-c90f-4d8d-8a8d-b96e48282afa', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 6, 22, 11, 44, 18, 205065, tzinfo=datetime.timezone.utc), content='Topic: benefits of a standing desk.', type='TextMessage'), TextMessage(id='608fb2c9-f258-49a4-8477-f6be7130027a', source='planner', models_usage=RequestUsage(prompt_tokens=42, completion_tokens=50), metadata={}, created_at=datetime.datetime(2026, 6, 22, 11, 44, 19, 461197, tzinfo=datetime.timezone.utc), content='1. How does using a standing desk impact overall productivity and focus in the workplace?  \n2. What are the physical health benefits associated with prolonged standing versus sitting?  \n3. How do standing desks affect energy levels and fatigue throughout the workday?', type='TextMessage'), TextMessage(id='32ad96b0-a8e9-4698-900c-f1d16a2e79f2', source='researcher', models_usage=RequestUsage(prompt_tokens=89, completion_tokens=166), metadata={}, created_at=

In [18]:
reviewer = AssistantAgent(
    name="reviewer",
    model_client=model_client,
    description="Reviews summaries and either approves them with 'APPROVE' or requests revisions with 'REVISE'.",
    system_message="You review summaries. If the summary is good, say APPROVE. If it needs changes, say REVISE.",
)

In [14]:
# === Extension 1 scaffold: custom selector_func ===
def force_planner_first(messages):
    # Return an agent name (str) to force a speaker, or None to let the model decide.
    if len(messages) <= 1:
        return "planner"
    return None

planner, researcher, writer = make_specialists()
team = SelectorGroupChat(
    participants=[planner, researcher, writer],
    model_client=model_client,
    selector_func=force_planner_first,
    termination_condition=TextMentionTermination("APPROVE") | MaxMessageTermination(8),
)
await Console(team.run_stream(task="Topic: benefits of a standing desk."))

---------- TextMessage (user) ----------
Topic: benefits of a standing desk.


ERROR:autogen_core:Error processing publish message for planner_4c4df02c-4218-409a-bde0-a90f0509377c/4c4df02c-4218-409a-bde0-a90f0509377c
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/openai/_base_client.py", line 1648, in request
    response = await self._send_request(
               ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/openai/_client.py", line 946, in _send_request
    return await self._send_with_auth_retry(request, stream=stream, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/openai/_client.py", line 924, in _send_with_auth_retry
    response = await super()._send_request(request, stream=stream, **kwargs)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/openai/_base_client.py", line 1571, in _send_request
    return await self._client.send(reques

RuntimeError: APIConnectionError: Connection error.
Traceback:
Traceback (most recent call last):

  File "/usr/local/lib/python3.12/dist-packages/openai/_base_client.py", line 1648, in request
    response = await self._send_request(
               ^^^^^^^^^^^^^^^^^^^^^^^^^

  File "/usr/local/lib/python3.12/dist-packages/openai/_client.py", line 946, in _send_request
    return await self._send_with_auth_retry(request, stream=stream, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

  File "/usr/local/lib/python3.12/dist-packages/openai/_client.py", line 924, in _send_with_auth_retry
    response = await super()._send_request(request, stream=stream, **kwargs)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

  File "/usr/local/lib/python3.12/dist-packages/openai/_base_client.py", line 1571, in _send_request
    return await self._client.send(request, stream=stream, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

In [11]:
# === Extension 2 scaffold: conditional edge (writer loops on REVISE) ===
builder = DiGraphBuilder()
builder.add_node(planner).add_node(researcher).add_node(writer).add_node(reviewer)
builder.add_edge(planner, researcher).add_edge(researcher, writer).add_edge(writer, reviewer)
# Conditional: reviewer -> writer only if "REVISE" appears; else the run ends.
builder.add_edge(reviewer, writer, condition=lambda msg: "REVISE" in msg.to_text())
graph = builder.build()
flow = GraphFlow(participants=builder.get_participants(), graph=graph)

In [20]:
# === Extension 2 scaffold: conditional edge (writer loops on REVISE) ===
# Re-create specialists to ensure they are clean for this flow
planner, researcher, writer = make_specialists()

# Define a final_report_agent to act as an explicit leaf node
final_report_agent = AssistantAgent(
    name="final_report_agent",
    model_client=model_client,
    description="Receives the final approved report.",
    system_message="You receive the final approved report and indicate task completion.",
)

builder = DiGraphBuilder()
builder.add_node(planner).add_node(researcher).add_node(writer).add_node(reviewer).add_node(final_report_agent)
builder.add_edge(planner, researcher).add_edge(researcher, writer).add_edge(writer, reviewer)
# Conditional: reviewer -> writer only if "REVISE" appears
builder.add_edge(reviewer, writer, condition=lambda msg: "REVISE" in msg.content.upper())
# Conditional: reviewer -> final_report_agent if "APPROVE" appears
builder.add_edge(reviewer, final_report_agent, condition=lambda msg: "APPROVE" in msg.content.upper())
graph = builder.build()

flow = GraphFlow(
    participants=builder.get_participants(),
    graph=graph,
    termination_condition=TextMentionTermination("APPROVE") | MaxMessageTermination(10) # Added a max message termination to prevent infinite loops
)

await Console(flow.run_stream(task="Topic: the benefits of meditation. Make sure the writer's first output is revised."))

---------- TextMessage (user) ----------
Topic: the benefits of meditation. Make sure the writer's first output is revised.
---------- TextMessage (planner) ----------
1. How does meditation impact mental health and emotional well-being?
2. What physiological effects does regular meditation practice have on the body?
3. In what ways can meditation enhance focus and cognitive performance?
---------- TextMessage (researcher) ----------
1. **Impact on Mental Health and Emotional Well-Being:**
   - Reduces symptoms of anxiety and depression.
   - Enhances emotional regulation and resilience.
   - Promotes a sense of calm and improved mood.
   - Increases self-awareness and mindfulness.

2. **Physiological Effects on the Body:**
   - Decreases blood pressure and heart rate.
   - Lowers levels of stress hormones (e.g., cortisol).
   - Improves immune system function.
   - Enhances brain structure and function (e.g., increased gray matter).

3. **Enhancement of Focus and Cognitive Performance

TaskResult(messages=[TextMessage(id='70834ff2-3d3b-4649-9130-89f474f78d24', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 6, 22, 11, 45, 45, 632955, tzinfo=datetime.timezone.utc), content="Topic: the benefits of meditation. Make sure the writer's first output is revised.", type='TextMessage'), TextMessage(id='5089f28a-32e8-412e-b99b-2223a94b5706', source='planner', models_usage=RequestUsage(prompt_tokens=51, completion_tokens=40), metadata={}, created_at=datetime.datetime(2026, 6, 22, 11, 45, 46, 757578, tzinfo=datetime.timezone.utc), content='1. How does meditation impact mental health and emotional well-being?\n2. What physiological effects does regular meditation practice have on the body?\n3. In what ways can meditation enhance focus and cognitive performance?', type='TextMessage'), TextMessage(id='acdb9830-c2b0-4d1d-bd1a-1ea6b961d79a', source='researcher', models_usage=RequestUsage(prompt_tokens=88, completion_tokens=169), metadata={}, created_a

In [1]:
# === Extension 3 scaffold: nest a team inside a node ===
from autogen_agentchat.teams import DiGraphBuilder, GraphFlow, RoundRobinGroupChat
# Removed: Explicitly import ChatAgent to try and resolve potential type mismatch issues
from autogen_agentchat.agents import AssistantAgent

# Re-create specialists to ensure they are clean for this flow
planner, researcher, writer = make_specialists()

# Define the fact_checker agent
fact_checker = AssistantAgent(
    name="fact_checker",
    model_client=model_client,
    description="Verifies facts provided by the researcher.",
    system_message="You are a fact-checker. Verify the information provided by the researcher.",
)

research_team = RoundRobinGroupChat(
    [researcher, fact_checker],
    termination_condition=MaxMessageTermination(4)
)

builder = DiGraphBuilder()
builder.add_node(planner).add_node(research_team).add_node(writer)
builder.add_edge(planner, research_team).add_edge(research_team, writer)
graph = builder.build()

flow = GraphFlow(
    # The participants for GraphFlow must include all top-level nodes from the builder
    # that can become active speakers, including the nested research_team itself.
    participants=[planner, research_team, writer],
    graph=graph,
    termination_condition=TextMentionTermination("APPROVE") | MaxMessageTermination(10)
)

await Console(flow.run_stream(task="Topic: the benefits of remote work. Research thoroughly and make sure all facts are checked."))

ModuleNotFoundError: No module named 'autogen_agentchat'

## Recap

You orchestrated one research team **five ways** and saw exactly how control flow differs:

* **Selector** — LLM picks the next speaker (dynamic).
* **Swarm** — agents hand off to each other (decentralised).
* **GraphFlow** — a fixed, auditable graph (deterministic).
* **Tools** — an agent delegates work to a function.
* **Semantic Kernel** — the same Sequential idea in the enterprise SDK.

Pair this with the decision matrix from the slides, then tackle the **capstone**: build your own delegating team in AutoGen Studio.

In [13]:
await model_client.close()
print("Client closed. On to the capstone!")

Client closed. On to the capstone!
